In [ ]:
import sys
from pathlib import Path

ROOT = Path.home() / "projects" / "complex_sabotage"
for p in [ROOT, ROOT / "plotting_notebooks", ROOT / "plotting_notebooks_protocols"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import json
import os
import sys
from typing import List, Optional, Dict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm
sys.path.append('..')
sys.path.append('../plotting_notebooks')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
)
import gc
from protocol_plotting_utils import add_task_specific_score_columns, load_protocol_intervention_transcripts, load_protocol_intervention_data
from utils.analysis_utils import analyze_all_tasks, extract_verdict
from utils.path_utils import resolve_path
from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
combined_results = load_in_combined_results(include_failed_attempts=True, include_failed_main_task=True)


In [ ]:
import json
from utils.path_utils import resolve_path

TOKEN_MSG = "input exceeds maximum token limit"
base = resolve_path("protocol_results/intervention_transcripts")

bad = []
for f in sorted(base.rglob("*.json")):
    try:
        d = json.load(open(f))
    except Exception as e:
        bad.append((f, f"unreadable: {e}"))
        continue

    reasons = []
    resp = d.get("monitoring_response", d.get("monitor_response"))
    if resp is None:
        reasons.append("no monitor response")
    elif isinstance(resp, str) and TOKEN_MSG in resp.lower():
        reasons.append("monitor hit token limit")
    if d.get("main_task_evaluation") is None:
        reasons.append("no main_task_evaluation")

    if reasons:
        bad.append((f, ", ".join(reasons)))

for f, why in bad:
    print(f"{why:45s} {f.relative_to(base)}")
print(f"\n{len(bad)} problem file(s)")

In [ ]:
# Test the updated functions
control_protocol_list = [ 'defer_to_trusted',]
protocol_results_list = []
intervention_data_list = []
for control_protocol in control_protocol_list:
    try:
        protocol_results = load_protocol_intervention_transcripts(protocol_name=control_protocol)
        intervention_data = load_protocol_intervention_data(protocol_name=control_protocol)
    except ValueError as e:
        print(f"Skipping {control_protocol}: {e}")
        continue
    protocol_results_list.append(protocol_results)
    intervention_data_list.append(intervention_data)
    
protocol_results = pd.concat(protocol_results_list)
intervention_data = pd.concat(intervention_data_list)

protocol_results["transcript_type"] = protocol_results["transcript_type"].str.replace("_no_thinking_time", "", regex=False)
intervention_data["transcript_type"] = intervention_data["transcript_type"].str.replace("_no_thinking_time", "", regex=False)

combined_results = combined_results[combined_results.task_name.isin(protocol_results.task_name)]
gc.collect()

In [ ]:
protocol_results['convo_length'] = protocol_results.conversation.apply(len)
protocol_results.sort_values('convo_length', ascending=True, inplace=True)
protocol_results.head()

In [ ]:
# Show some basic stats
print(f"Protocol transcripts: {len(protocol_results)} records")
print(f"Protocol data: {len(intervention_data)} records")

# Show columns for each dataset
print("\nProtocol transcripts columns:")
print(protocol_results.columns.tolist())

print("\nIntervention data columns:")
print(intervention_data.columns.tolist())

In [ ]:
protocol_results.columns
protocol_results_and_data = pd.merge(protocol_results, intervention_data, on=['task_name', 'transcript_type', 'protocol_name', 'main_agent', 'monitor_agent', 'file_name'], how='inner')

In [ ]:
protocol_results_and_data['turn_by_turn_suspicion_scores'] = protocol_results_and_data.raw_data.apply(lambda x: [y['suspicion_score'] for y in x['control_metadata']])
plt.hist(protocol_results_and_data[protocol_results_and_data.transcript_type == 'transcripts_no_secondary_task'].turn_by_turn_suspicion_scores.sum(), bins=30, density=True, alpha = 0.5, label = 'no secondary task')
plt.hist(protocol_results_and_data[protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task'].turn_by_turn_suspicion_scores.sum(), bins=30, density=True, alpha = 0.5, label = 'secondary task')
plt.legend()
plt.title('Turn by turn suspicion scores')
plt.show()

plt.hist(protocol_results_and_data[protocol_results_and_data.transcript_type == 'transcripts_no_secondary_task'].turn_by_turn_suspicion_scores.apply(lambda x: np.max(x)), bins=30, density=True, alpha = 0.5, label = 'no secondary task')
plt.hist(protocol_results_and_data[protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task'].turn_by_turn_suspicion_scores.apply(lambda x: np.max(x)), bins=30, density=True, alpha = 0.5, label = 'secondary task')
plt.legend()
plt.title('Max turn by turn suspicion scores')
plt.show()

In [ ]:
main_agent = 'gemma-4-31B-it-FP8-block'
trusted_agent = 'gemma-4-31B-it-FP8-block'
for control_protocol in control_protocol_list:
    plt.figure(figsize=(10, 5))
    for task_name in protocol_results_and_data.task_name.unique():
        intervention_threshold = []
        main_task_evaluation = []
        intervention_threshold.append(0)
        main_task_evaluation.append(combined_results[(combined_results.task_name == task_name) & (combined_results.main_agent == trusted_agent)].main_task_evaluation.mean())
        filtered_protocol = protocol_results_and_data[(protocol_results_and_data.protocol_name == control_protocol) & (protocol_results_and_data.task_name == task_name) & (protocol_results_and_data.main_agent == main_agent)]
        for threshold in sorted(filtered_protocol.suspicion_threshold.unique()):
            filtered_protocol_threshold = filtered_protocol[filtered_protocol.suspicion_threshold == threshold]
            intervention_threshold.append(threshold)
            main_task_evaluation.append(filtered_protocol_threshold.main_task_evaluation.mean())
        intervention_threshold.append(100)
        main_task_evaluation.append(combined_results[(combined_results.task_name == task_name) & (combined_results.main_agent == main_agent)].main_task_evaluation.mean())
        plt.plot(intervention_threshold, main_task_evaluation, label = task_name)
    plt.title(f"{control_protocol} main task evaluation")
    plt.legend()
    plt.show()

In [ ]:
protocol_results_and_data.suspicion_threshold.unique()

In [ ]:
# LET'S ONLY LOOK AT MALIGN TRANSCRIPTs
main_agent = 'gemma-4-31B-it-FP8-block'
trusted_agent = 'gemma-4-31B-it-FP8-block'
for control_protocol in control_protocol_list:
    plt.figure(figsize=(10, 5))
    concat_list = []
    for task_name in protocol_results_and_data.task_name.unique():
        threshold_dict = {'task_name': task_name}
        threshold_dict[0] = combined_results[(combined_results.task_name == task_name) & (combined_results.main_agent == trusted_agent) & (combined_results.transcript_type != 'transcripts_no_secondary_task')].main_task_success.mean()
        filtered_protocol = protocol_results_and_data[(protocol_results_and_data.protocol_name == control_protocol) & (protocol_results_and_data.task_name == task_name) & (protocol_results_and_data.main_agent == main_agent) & (protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task')]
        for threshold in sorted(filtered_protocol.suspicion_threshold.unique()):
            filtered_protocol_threshold = filtered_protocol[filtered_protocol.suspicion_threshold == threshold]
            threshold_dict[threshold] = filtered_protocol_threshold.main_task_success.mean()
        threshold_dict[100] = combined_results[(combined_results.task_name == task_name) & (combined_results.main_agent == main_agent) & (combined_results.transcript_type != 'transcripts_no_secondary_task')].main_task_success.mean()
        concat_list.append(threshold_dict)
        
for name in ["combined_results", "protocol_results", "intervention_data"]:
    obj = globals().get(name)
    print(f"{name:20s}", None if obj is None else obj.shape)

# do the keys used to match these frames actually overlap?
for col in ["task_name", "main_agent", "monitor_agent", "transcript_type"]:
    a = set(combined_results[col].unique()) if col in combined_results else set()
    b = set(protocol_results[col].unique()) if col in protocol_results else set()
    print(f"\n{col}\n  combined only: {sorted(a - b)[:5]}\n  protocol only: {sorted(b - a)[:5]}\n  shared: {len(a & b)}")
        
    concat_df = pd.DataFrame(concat_list)


In [ ]:
concat_df

In [ ]:
# Create a single bar chart for all tasks with sorted thresholds
plt.figure(figsize=(15, 8))

# Convert threshold columns to numeric for proper sorting
threshold_cols = [col for col in concat_df.columns if col != 'task_name']
numeric_thresholds = [float(col) if isinstance(col, (int, float)) else float(col) for col in threshold_cols]
sorted_indices = np.argsort(numeric_thresholds)
sorted_threshold_cols = [threshold_cols[i] for i in sorted_indices]

print(concat_df.shape)
print(concat_df.columns.tolist())
print("threshold_cols:", threshold_cols)


# Set up the plot
ax = plt.subplot(111)
bar_width = 0.7 / len(sorted_threshold_cols)
task_names = concat_df['task_name'].tolist()

# Plot the bars for each threshold
for i, col in enumerate(sorted_threshold_cols):
    # Get values for this threshold
    values = concat_df[col].values
    # Position on the x-axis
    x_pos = np.arange(len(task_names)) + i * bar_width
    # Plot the bars
    bars = ax.bar(x_pos, values, width=bar_width, label=str(col))
    
    # Add value labels on the bars
    for bar, val in zip(bars, values):
        if not pd.isna(val):  # Only add labels for non-NaN values
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{val:.2f}', ha='center', va='bottom', rotation=90, fontsize=8)

# Set labels and title
ax.set_xlabel('Task Name')
ax.set_ylabel('Main Task Success Rate')
ax.set_title('Task Performance by Threshold')
ax.set_xticks(np.arange(len(task_names)) + (len(sorted_threshold_cols) - 1) * bar_width / 2)
ax.set_xticklabels(task_names, rotation=45, ha='right')
ax.set_ylim(0, 1.1)  # Set y-axis limit with some margin for labels
ax.legend(title='Threshold')

plt.tight_layout()
plt.show()


In [ ]:
# Alternative visualization: heatmap
import seaborn as sns

# Reindex the dataframe with task_name as index
heatmap_df = concat_df.set_index('task_name')

# Order the columns numerically
ordered_cols = sorted([col for col in heatmap_df.columns], key=lambda x: float(x) if isinstance(x, (int, float)) else float(x))
heatmap_df = heatmap_df[ordered_cols]

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_df, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=.5)
plt.title('Task Performance by Threshold (Heatmap)')
plt.tight_layout()
plt.show()
